In [ ]:
def diagnose_background_distribution_counts(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    clonotypes_to_plot: list[str] | None = None,
    n_clonotypes: int = 12,
    min_size: int = 5,
    min_prop_threshold: float = 0.05,
    control_peptide: str | None = None,
    out_pdf: Path | None = None,
):
    """
    Makes a plot for a clonotype, comparing the raw dex counts against a Poisson
    distribution, negative binomial distribution, and just standard percentile thresholds
    """
    from scipy import stats
    
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    
    if clonotypes_to_plot is None:
        all_cts = list(keep)
        if len(all_cts) > n_clonotypes:
            step = len(all_cts) // n_clonotypes
            clonotypes_to_plot = [all_cts[i * step] for i in range(n_clonotypes)]
        else:
            clonotypes_to_plot = all_cts
    
    control_idx = None
    if control_peptide is not None and control_peptide in peptides:
        control_idx = peptides.index(control_peptide)
    
    summary_rows = []
    
    if out_pdf is not None:
        out_pdf = Path(out_pdf)
        out_pdf.parent.mkdir(parents=True, exist_ok=True)
        pdf = PdfPages(out_pdf)
    else:
        pdf = None
    
    for ct in clonotypes_to_plot:
        g = df2[df2[ct_col] == ct]
        X = g[markers].to_numpy(dtype=float)
        n_cells = X.shape[0]
        
        mean_counts = X.mean(axis=0)
        total = mean_counts.sum()
        mean_props = mean_counts / total if total > 0 else np.zeros_like(mean_counts)
        
        background_mask = mean_props < min_prop_threshold
        candidate_mask = mean_props >= min_prop_threshold
        
        background_peptides = [p for p, is_bg in zip(peptides, background_mask) if is_bg]
        candidate_peptides = [p for p, is_cand in zip(peptides, candidate_mask) if is_cand]
        
        background_counts_pooled = X[:, background_mask].flatten()
        
        if control_idx is not None:
            control_counts = X[:, control_idx]
            control_mean = np.mean(control_counts)
        else:
            control_counts = None
            control_mean = np.nan
        
        bg_mean_count = np.mean(background_counts_pooled) if len(background_counts_pooled) > 0 else np.nan
        bg_var_count = np.var(background_counts_pooled) if len(background_counts_pooled) > 0 else np.nan
        bg_std_count = np.std(background_counts_pooled) if len(background_counts_pooled) > 0 else np.nan
        
        dispersion_index = bg_var_count / bg_mean_count if bg_mean_count > 0 else np.nan
        
        if bg_mean_count > 0:
            poisson_95 = stats.poisson.ppf(0.95, bg_mean_count)
            poisson_99 = stats.poisson.ppf(0.99, bg_mean_count)
        else:
            poisson_95, poisson_99 = np.nan, np.nan
        
        if len(background_counts_pooled) > 0:
            percentile_95 = np.percentile(background_counts_pooled, 95)
            percentile_99 = np.percentile(background_counts_pooled, 99)
        else:
            percentile_95, percentile_99 = np.nan, np.nan
        
        if bg_mean_count > 0 and bg_var_count > bg_mean_count:
            alpha_nb = bg_mean_count**2 / (bg_var_count - bg_mean_count) if bg_var_count > bg_mean_count else np.inf
            if np.isfinite(alpha_nb) and alpha_nb > 0:
                n_nb = alpha_nb
                p_nb = alpha_nb / (alpha_nb + bg_mean_count)
                nb_95 = stats.nbinom.ppf(0.95, n_nb, p_nb)
                nb_99 = stats.nbinom.ppf(0.99, n_nb, p_nb)
            else:
                nb_95, nb_99 = np.nan, np.nan
        else:
            nb_95, nb_99 = np.nan, np.nan
            alpha_nb = np.nan
        
        summary_rows.append({
            "clonotype": ct,
            "n_cells": n_cells,
            "n_background_peptides": int(background_mask.sum()),
            "n_candidate_peptides": int(candidate_mask.sum()),
            "n_background_observations": len(background_counts_pooled),
            "bg_mean_count": bg_mean_count,
            "bg_var_count": bg_var_count,
            "bg_std_count": bg_std_count,
            "dispersion_index": dispersion_index,
            "poisson_95": poisson_95,
            "poisson_99": poisson_99,
            "percentile_95": percentile_95,
            "percentile_99": percentile_99,
            "nb_alpha": alpha_nb,
            "nb_95": nb_95,
            "nb_99": nb_99,
            "control_mean_count": control_mean,
            "background_peptides": background_peptides,
            "candidate_peptides": candidate_peptides,
        })
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 11))
        
        ax = axes[0, 0]
        if len(background_counts_pooled) > 0:
            max_count = int(np.max(background_counts_pooled))
            bins = np.arange(-0.5, max_count + 1.5, 1)
            
            ax.hist(background_counts_pooled, bins=bins, density=True, alpha=0.7, 
                    color="steelblue", edgecolor="white", label="Observed")
            
            if bg_mean_count > 0:
                x_poisson = np.arange(0, max(max_count + 1, int(poisson_99) + 3))
                y_poisson = stats.poisson.pmf(x_poisson, bg_mean_count)
                ax.plot(x_poisson, y_poisson, 'r-', lw=2, marker='o', markersize=4,
                        label=f'Poisson(λ={bg_mean_count:.2f})')
                
                if np.isfinite(nb_95):
                    y_nb = stats.nbinom.pmf(x_poisson, n_nb, p_nb)
                    ax.plot(x_poisson, y_nb, 'g--', lw=2, marker='s', markersize=4,
                            label=f'NegBinom(α={alpha_nb:.2f})')
            
            ax.axvline(poisson_95, color="red", ls="--", lw=1.5, alpha=0.7, label=f'Poisson 95%: {poisson_95:.0f}')
            ax.axvline(percentile_95, color="orange", ls=":", lw=2, alpha=0.9,label=f'Empirical 95%: {percentile_95:.0f}')
            
            ax.legend(fontsize=8, loc="upper right")
            ax.set_xlim(-0.5, min(max_count + 2, 50))
        
        ax.set_xlabel("Raw count")
        ax.set_ylabel("Density")
        ax.set_title(f"Background counts (n={len(background_counts_pooled)})")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        ax = axes[0, 1]
        
        peptide_means = X.mean(axis=0)
        peptide_vars = X.var(axis=0)
        
        colors = ['steelblue' if background_mask[i] else 'darkred' for i in range(len(peptides))]
        ax.scatter(peptide_means, peptide_vars, c=colors, s=60, alpha=0.7, edgecolors='white')
        
        for i, pep in enumerate(peptides):
            ax.annotate(pep, (peptide_means[i], peptide_vars[i]), fontsize=6, alpha=0.8, xytext=(3, 3), textcoords='offset points')
        
        max_val = max(peptide_means.max(), peptide_vars.max()) * 1.1
        ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Poisson (var = mean)')
        
        ax.fill_between([0, max_val], [0, max_val], [0, max_val * 5], 
                        alpha=0.1, color='orange', label='Overdispersed region')
        
        ax.set_xlabel("Mean count")
        ax.set_ylabel("Variance")
        ax.set_title(f"Variance vs Mean (Dispersion index: {dispersion_index:.2f})")
        ax.legend(fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_xlim(0, max_val)
        ax.set_ylim(0, max_val * 1.5)
        
        if dispersion_index < 1.5:
            interp = "≈ Poisson (good fit)"
            interp_color = "green"
        elif dispersion_index < 3:
            interp = "Mild overdispersion"
            interp_color = "orange"
        else:
            interp = "Strong overdispersion\n(use NegBinom or Percentile)"
            interp_color = "red"
        ax.text(0.05, 0.95, interp, transform=ax.transAxes, fontsize=10, 
                color=interp_color, fontweight='bold', va='top')
        
        ax = axes[1, 0]
        
        threshold_names = ['Poisson\n95%', 'Poisson\n99%', 'Percentile\n95%', 'Percentile\n99%', 'NegBinom\n95%', 'NegBinom\n99%', 'Control\nmean']
        threshold_values = [poisson_95, poisson_99, percentile_95, percentile_99, nb_95, nb_99, control_mean]
        threshold_colors = ['red', 'darkred', 'orange', 'darkorange', 'green', 'darkgreen', 'purple']
        
        valid_mask = [np.isfinite(v) for v in threshold_values]
        valid_names = [n for n, v in zip(threshold_names, valid_mask) if v]
        valid_values = [v for v, m in zip(threshold_values, valid_mask) if m]
        valid_colors = [c for c, m in zip(threshold_colors, valid_mask) if m]
        
        bars = ax.bar(valid_names, valid_values, color=valid_colors, alpha=0.7, edgecolor='white')
        
        for bar, val in zip(bars, valid_values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}', ha='center', va='bottom', fontsize=9)
        
        ax.set_ylabel("Count threshold")
        ax.set_title("Threshold Comparison")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        ax = axes[1, 1]
        xs = np.arange(len(peptides))
        
        peptide_mean_counts = X.mean(axis=0)
        peptide_std_counts = X.std(axis=0)
        
        colors = []
        for k in range(len(peptides)):
            if candidate_mask[k]:
                if peptide_mean_counts[k] > percentile_95:
                    colors.append("darkred")
                else:
                    colors.append("darkorange")
            else:
                if control_idx is not None and k == control_idx:
                    colors.append("purple")
                else:
                    colors.append("steelblue")
        
        ax.bar(xs, peptide_mean_counts, yerr=peptide_std_counts, color=colors, 
               alpha=0.7, capsize=2, edgecolor='white')
        
        # Thresholds
        if np.isfinite(poisson_95):
            ax.axhline(poisson_95, color="red", ls="--", lw=1.5, label=f'Poisson 95%: {poisson_95:.1f}')
        if np.isfinite(percentile_95):
            ax.axhline(percentile_95, color="orange", ls=":", lw=2, label=f'Percentile 95%: {percentile_95:.1f}')
        if np.isfinite(nb_95):
            ax.axhline(nb_95, color="green", ls="-.", lw=1.5, label=f'NegBinom 95%: {nb_95:.1f}')
        
        ax.set_xticks(xs)
        ax.set_xticklabels(peptides, rotation=90, fontsize=7)
        ax.set_ylabel("Mean count ± SD")
        ax.set_title(f"Per-peptide counts (n_cells={n_cells})")
        ax.legend(fontsize=7, loc="upper right")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        fig.suptitle(f"{ct}\nBackground: {', '.join(background_peptides[:4])}{'...' if len(background_peptides) > 4 else ''}", 
                     fontsize=10, y=0.98)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        
        if pdf is not None:
            pdf.savefig(fig)
            plt.close(fig)
        else:
            plt.show()
    
    if pdf is not None:
        pdf.close()
        print(f"Saved: {out_pdf}")
    
    summary_df = pd.DataFrame(summary_rows)
    return summary_df

In [ ]:
def diagnose_background_distribution(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    clonotypes_to_plot: list[str] | None = None,
    n_clonotypes: int = 12,
    min_size: int = 5,
    min_prop_threshold: float = 0.05,
    control_peptide: str | None = None,
    out_pdf: Path | None = None,
):
    """
    Looks at an individual clonotype comparing against a normal distribution
    """
    from scipy import stats
    
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    
    if clonotypes_to_plot is None:
        all_cts = list(keep)
        if len(all_cts) > n_clonotypes:
            step = len(all_cts) // n_clonotypes
            clonotypes_to_plot = [all_cts[i * step] for i in range(n_clonotypes)]
        else:
            clonotypes_to_plot = all_cts
    
    control_idx = None
    if control_peptide is not None and control_peptide in peptides:
        control_idx = peptides.index(control_peptide)
    
    summary_rows = []
    
    if out_pdf is not None:
        out_pdf = Path(out_pdf)
        out_pdf.parent.mkdir(parents=True, exist_ok=True)
        pdf = PdfPages(out_pdf)
    else:
        pdf = None
    
    for ct in clonotypes_to_plot:
        g = df2[df2[ct_col] == ct]
        X = g[markers].to_numpy(dtype=float)
        n_cells = X.shape[0]
        

        row_sums = X.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        P = X / row_sums
        
        mean_counts = X.mean(axis=0)
        total = mean_counts.sum()
        mean_props = mean_counts / total if total > 0 else np.zeros_like(mean_counts)
        
        background_mask = mean_props < min_prop_threshold
        candidate_mask = mean_props >= min_prop_threshold
        
        background_peptides = [p for p, is_bg in zip(peptides, background_mask) if is_bg]
        candidate_peptides = [p for p, is_cand in zip(peptides, candidate_mask) if is_cand]
        
        background_props_pooled = P[:, background_mask].flatten()
        
        if control_idx is not None:
            control_props = P[:, control_idx]
        else:
            control_props = None
        
        bg_mean = np.mean(background_props_pooled) if len(background_props_pooled) > 0 else np.nan
        bg_std = np.std(background_props_pooled) if len(background_props_pooled) > 0 else np.nan
        bg_median = np.median(background_props_pooled) if len(background_props_pooled) > 0 else np.nan
        
        if len(background_props_pooled) >= 20:
            sample = background_props_pooled if len(background_props_pooled) <= 5000 else np.random.choice(background_props_pooled, 5000, replace=False)
            shapiro_stat, shapiro_p = stats.shapiro(sample)
        else:
            shapiro_stat, shapiro_p = np.nan, np.nan
        
        summary_rows.append({
            "clonotype": ct,
            "n_cells": n_cells,
            "n_background_peptides": int(background_mask.sum()),
            "n_candidate_peptides": int(candidate_mask.sum()),
            "n_background_observations": len(background_props_pooled),
            "background_mean": bg_mean,
            "background_std": bg_std,
            "background_median": bg_median,
            "shapiro_stat": shapiro_stat,
            "shapiro_p": shapiro_p,
            "background_peptides": background_peptides,
            "candidate_peptides": candidate_peptides,
        })
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        ax = axes[0, 0]
        if len(background_props_pooled) > 0:
            ax.hist(background_props_pooled, bins=50, density=True, alpha=0.7, color="steelblue", edgecolor="white")
            
            x_range = np.linspace(0, max(background_props_pooled.max(), min_prop_threshold), 100)
            if bg_std > 0:
                normal_pdf = stats.norm.pdf(x_range, bg_mean, bg_std)
                ax.plot(x_range, normal_pdf, 'r-', lw=2, label=f'Normal fit (μ={bg_mean:.4f}, σ={bg_std:.4f})')
            
            ax.axvline(min_prop_threshold, color="red", ls="--", lw=1.5, label=f'{min_prop_threshold:.0%} threshold')
            ax.axvline(bg_mean, color="green", ls="-", lw=1.5, alpha=0.7, label=f'Mean: {bg_mean:.4f}')
            if bg_std > 0:
                ax.axvline(bg_mean + 2*bg_std, color="orange", ls=":", lw=1.5, label=f'Mean + 2SD: {bg_mean + 2*bg_std:.4f}')
            
            ax.legend(fontsize=8)
        ax.set_xlabel("Proportion (per-cell)")
        ax.set_ylabel("Density")
        ax.set_title(f"Background peptides pooled (n={len(background_props_pooled)})")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        ax = axes[0, 1]
        if control_props is not None:
            ax.hist(control_props, bins=30, density=True, alpha=0.7, color="purple", edgecolor="white")
            ctrl_mean = np.mean(control_props)
            ctrl_std = np.std(control_props)
            ax.axvline(ctrl_mean, color="green", ls="-", lw=1.5, label=f'Mean: {ctrl_mean:.4f}')
            ax.axvline(min_prop_threshold, color="red", ls="--", lw=1.5, label=f'{min_prop_threshold:.0%} threshold')
            
            if len(background_props_pooled) > 0:
                ax.hist(background_props_pooled, bins=50, density=True, alpha=0.3, color="steelblue", label="All background")
            
            ax.set_title(f"Control peptide ({control_peptide}) vs background")
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, "No control peptide specified", ha="center", va="center", transform=ax.transAxes)
            ax.set_title("Control peptide distribution")
        ax.set_xlabel("Proportion (per-cell)")
        ax.set_ylabel("Density")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        ax = axes[1, 0]
        if len(background_props_pooled) >= 10:
            stats.probplot(background_props_pooled, dist="norm", plot=ax)
            ax.set_title(f"QQ-plot (Shapiro p={shapiro_p:.4f})" if not np.isnan(shapiro_p) else "QQ-plot")
        else:
            ax.text(0.5, 0.5, "Not enough data for QQ-plot", ha="center", va="center", transform=ax.transAxes)
            ax.set_title("QQ-plot")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        ax = axes[1, 1]
        xs = np.arange(len(peptides))
        
        peptide_means = P.mean(axis=0)
        peptide_stds = P.std(axis=0)
        
        colors = []
        for k in range(len(peptides)):
            if candidate_mask[k]:
                if bg_std > 0 and peptide_means[k] > bg_mean + 2 * bg_std:
                    colors.append("darkred")
                else:
                    colors.append("darkorange")
            else:
                if control_idx is not None and k == control_idx:
                    colors.append("purple")
                else:
                    colors.append("steelblue")
        
        ax.bar(xs, peptide_means, yerr=peptide_stds, color=colors, alpha=0.7, capsize=2)
        ax.axhline(min_prop_threshold, color="red", ls="--", lw=1, label=f'{min_prop_threshold:.0%} threshold')
        if bg_std > 0:
            ax.axhline(bg_mean + 2*bg_std, color="orange", ls=":", lw=1.5, label=f'BG mean + 2SD')
        ax.axhline(bg_mean, color="green", ls="-", lw=1, alpha=0.7, label=f'BG mean')
        
        ax.set_xticks(xs)
        ax.set_xticklabels(peptides, rotation=90, fontsize=7)
        ax.set_ylabel("Mean proportion ± SD")
        ax.set_title(f"Per-peptide summary (n_cells={n_cells})")
        ax.legend(fontsize=7, loc="upper right")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        fig.suptitle(f"{ct}\nBackground: {', '.join(background_peptides[:5])}{'...' if len(background_peptides) > 5 else ''}", 
                     fontsize=10, y=0.98)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        
        if pdf is not None:
            pdf.savefig(fig)
            plt.close(fig)
        else:
            plt.show()
    
    if pdf is not None:
        pdf.close()
        print(f"Saved: {out_pdf}")
    
    return pd.DataFrame(summary_rows)

In [ ]:
bg_summary_b10br = diagnose_background_distribution(
    df=df_b10br_combined,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    n_clonotypes=12,
    min_size=5,
    min_prop_threshold=0.05,
    control_peptide="EEEPVKKI",
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/background_distribution_diagnostic.pdf"),
)

bg_summary_balbc = diagnose_background_distribution(
    df=df_balbc_combined,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    n_clonotypes=12,
    min_size=5,
    min_prop_threshold=0.05,
    control_peptide="SYFPEITHI",
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/background_distribution_diagnostic.pdf"),
)

print(bg_summary_b10br[["clonotype", "n_cells", "background_mean", "background_std", "shapiro_p"]])

In [ ]:
bg_counts_summary_b10br = diagnose_background_distribution_counts(
    df=df_b10br_combined,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    n_clonotypes=20,
    min_size=5,
    min_prop_threshold=0.05,
    control_peptide="EEEPVKKI",
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/background_counts_diagnostic.pdf"),
)

bg_counts_summary_balbc = diagnose_background_distribution_counts(
    df=df_balbc_combined,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    n_clonotypes=20,
    min_size=5,
    min_prop_threshold=0.05,
    control_peptide="SYFPEITHI",
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/background_counts_diagnostic.pdf"),
)

print("B10BR Dispersion Summary:")
print(bg_counts_summary_b10br[["clonotype", "n_cells", "bg_mean_count", "bg_var_count", "dispersion_index", "poisson_95", "percentile_95", "nb_95"]].to_string())